# Lesson 3 — Tool design

**You will:** see how an LLM 'knows' which tool to call, and learn that tool descriptions are part of the prompt.

[Open in Colab](https://colab.research.google.com/github/richey-malhotra/barebear/blob/main/lessons/03-tool-design/lesson.ipynb)
[Read the lesson narrative](./lesson.md)
[Back to syllabus](../README.md)

## The big idea

You handed the agent a tool in Lesson 2. The agent picked it. *How did the model know it existed?*

It didn't, until you told it. Every tool gets converted into a *schema* — a small JSON object describing the function and its parameters. That schema goes to the model on every turn. The model reads it and decides whether to call the tool.

**Tool descriptions are part of the prompt.** Vague descriptions confuse the model. Confusing parameter names cause garbage arguments. Tool design is prompt engineering wearing a different hat.

## Setup

Run the next two cells once per session. They install barebear and set your OpenRouter key (free tier — get one at <https://openrouter.ai>).

In [ ]:
%pip install -q "barebear[openai]"

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key (sk-or-...): ")
print("Key set. First 8 characters:", os.environ["OPENROUTER_API_KEY"][:8] + "...")

## Three confusable tools

First, three tools with the same vague description. Watch the model flounder.

In [ ]:
from barebear import Bear, Task, Tool, OpenRouterModel

def search_web(query: str) -> str:
    return f"Web results for '{query}': [recent news, blog posts, reddit threads]"

def search_docs(query: str) -> str:
    return f"Docs results for '{query}': [API reference, guides, examples]"

def search_database(query: str) -> str:
    return f"DB results for '{query}': [10 rows from products table]"

bad_tools = [
    Tool(name="search_web", fn=search_web, description="Search for things"),
    Tool(name="search_docs", fn=search_docs, description="Search for things"),
    Tool(name="search_database", fn=search_database, description="Search for things"),
]

bear = Bear(model=OpenRouterModel(), tools=bad_tools)
bear.run(Task(goal="Find the API spec for the /users endpoint."), trace=True)

## Now write decent descriptions

In [ ]:
good_tools = [
    Tool(name="search_web", fn=search_web,
         description="Search the public web (news, blogs, social media). Use for current events or external opinions."),
    Tool(name="search_docs", fn=search_docs,
         description="Search internal API and developer documentation. Use for technical references and code examples."),
    Tool(name="search_database", fn=search_database,
         description="Query the application's product database. Use for facts about products, customers, or orders."),
]

bear = Bear(model=OpenRouterModel(), tools=good_tools)
bear.run(Task(goal="Find the API spec for the /users endpoint."), trace=True)

## See the schema

BareBear converts each tool into a JSON schema before sending it to the model. Run the cell to see one.

In [ ]:
import json

t = good_tools[1]   # search_docs
print(json.dumps(t.to_openai_schema(), indent=2))

## Exercise

Add a `summarize(text)` tool. Write its description three different ways: cryptic, verbose, and Goldilocks. Which version produces the most reliable behaviour when the agent has all four tools?

In [ ]:
def summarize(text: str) -> str:
    return text[:200]

# Try three different descriptions and compare.

## What's next

[Lesson 4 →](../04-state-and-memory/lesson.ipynb)